# CMPE 58T — Turkish Dependency Parsing
## 02 — Baseline: BERTurk + SuPar CRF2oDependencyParser

**Key design decisions:**
- Per-treebank LR: BOUN=1e-3, IMST=2e-3 (empirically determined)
- UAS/LAS from `parser.evaluate()` — not manual loop (avoids tensor-indexing bugs)
- `run_inference()` uses `parser.predict()` to write CoNLL-U files
- `force_rerun=True` in eval cell to avoid stale cache after retraining

---
## 0 — Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob

PROJECT_DIR  = "/content/drive/MyDrive/58T - Application Project"
DATA_DIR     = os.path.join(PROJECT_DIR, "data")
VOCAB_DIR    = os.path.join(PROJECT_DIR, "vocab")
MODEL_DIR    = os.path.join(PROJECT_DIR, "models")
PRED_DIR     = os.path.join(PROJECT_DIR, "predictions")
RESULTS_DIR  = os.path.join(PROJECT_DIR, "results")
for d in [DATA_DIR, VOCAB_DIR, MODEL_DIR, PRED_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

def get_splits(tb_name):
    folder = os.path.join(DATA_DIR, f"UD_Turkish-{tb_name}")
    files  = sorted(glob.glob(os.path.join(folder, "*.conllu")))
    splits = {}
    for f in files:
        name = os.path.basename(f)
        if   "train" in name: splits["train"] = f
        elif "dev"   in name: splits["dev"]   = f
        elif "test"  in name: splits["test"]  = f
    return splits

BOUN = get_splits("BOUN")
IMST = get_splits("IMST")
PUD  = get_splits("PUD")

MODEL_PATHS = {
    "BOUN": os.path.join(MODEL_DIR, "baseline_boun.pt"),
    "IMST": os.path.join(MODEL_DIR, "baseline_imst.pt"),
}

print("Treebank splits:")
for tb, splits in [("BOUN", BOUN), ("IMST", IMST), ("PUD", PUD)]:
    for split, path in splits.items():
        print(f"  {tb} {split:<6} -> {path}")
print()
print("Model paths:")
for tb, path in MODEL_PATHS.items():
    status = "(exists — will skip training)" if os.path.exists(path) else "(new — will train)"
    print(f"  {tb}: {os.path.basename(path)}  {status}")
print()
print(f"Results dir  : {RESULTS_DIR}")
print(f"Predictions  : {PRED_DIR}")


Mounted at /content/drive
Treebank splits:
  BOUN dev    -> /content/drive/MyDrive/58T - Application Project/data/UD_Turkish-BOUN/tr_boun-ud-dev.conllu
  BOUN test   -> /content/drive/MyDrive/58T - Application Project/data/UD_Turkish-BOUN/tr_boun-ud-test.conllu
  BOUN train  -> /content/drive/MyDrive/58T - Application Project/data/UD_Turkish-BOUN/tr_boun-ud-train.conllu
  IMST dev    -> /content/drive/MyDrive/58T - Application Project/data/UD_Turkish-IMST/tr_imst-ud-dev.conllu
  IMST test   -> /content/drive/MyDrive/58T - Application Project/data/UD_Turkish-IMST/tr_imst-ud-test.conllu
  IMST train  -> /content/drive/MyDrive/58T - Application Project/data/UD_Turkish-IMST/tr_imst-ud-train.conllu
  PUD test   -> /content/drive/MyDrive/58T - Application Project/data/UD_Turkish-PUD/tr_pud-ud-test.conllu

Model paths:
  BOUN: baseline_boun.pt  (exists — will skip training)
  IMST: baseline_imst.pt  (exists — will skip training)

Results dir  : /content/drive/MyDrive/58T - Application Project

---
## 1 — Install & Patch

**Must be run at the start of every session.**

In [2]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install "transformers==4.35.2" -q
!pip install supar -q
!pip install conllu -q
!pip install scipy -q

import os
PARSER_PY = "/usr/local/lib/python3.12/dist-packages/supar/parsers/parser.py"
OLD = 'state = torch.load(path if os.path.exists(path) else download(supar.MODEL[src].get(path, path), reload=reload))'
NEW = 'state = torch.load(path if os.path.exists(path) else download(supar.MODEL[src].get(path, path), reload=reload), weights_only=False)'
with open(PARSER_PY) as f: src = f.read()
if OLD in src:
    with open(PARSER_PY, 'w') as f: f.write(src.replace(OLD, NEW))
    print("Patch applied: torch.load weights_only=False")
else:
    print("Patch already applied")

from transformers import AdamW
import torch, transformers
print(f"transformers : {transformers.__version__}")
print(f"PyTorch      : {torch.__version__}")
print(f"GPU          : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — switch runtime!'}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.5/123.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 81.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.35.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.7/773.7 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 28.6 MB/s e

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


---
## 2 — Hyperparameters

In [3]:
BERTURK_NAME = "dbmdz/bert-base-turkish-uncased"
HIDDEN_SIZE  = 800
DROPOUT      = 0.33
BATCH_SIZE   = 5000
BERT_LR      = 2e-5
WEIGHT_DECAY = 0.01
EPS          = 1e-8
WARMUP       = 0.1
MAX_EPOCHS   = 30
PATIENCE     = 10
SEED         = 42

# Per-treebank LR — empirically determined:
#   BOUN: 1e-3 (2e-3 collapses label head on BOUN)
#   IMST: 2e-3 (1e-3 underfits on the smaller IMST set)
LR_PER_TB = {
    "BOUN": 1e-3,
    "IMST": 2e-3,
}

# Must be applied via parser.args.update() after every .load() call
SUPAR_EXTRA = dict(
    mbr=True, partial=False, tree=False,
    proj=False, punct=False, update_steps=1, clip=5.0
)

print(f"BERT_LR={BERT_LR}  MAX_EPOCHS={MAX_EPOCHS}  PATIENCE={PATIENCE}  BATCH={BATCH_SIZE}")
print(f"LR_PER_TB: BOUN={LR_PER_TB['BOUN']}  IMST={LR_PER_TB['IMST']}")


BERT_LR=2e-05  MAX_EPOCHS=30  PATIENCE=10  BATCH=5000
LR_PER_TB: BOUN=0.001  IMST=0.002


---
## 3 — Helpers

`run_inference` uses `parser.predict()` — guaranteed correct predictions.
`evaluate_and_save` uses `parser.evaluate()` for UAS/LAS and `score_predictions` only for the per-sentence npy arrays needed by McNemar's tests.

In [4]:
import torch
import numpy as np
import conllu as conllu_lib
import json
from collections import defaultdict
from supar import CRF2oDependencyParser

def load_gold(conllu_path):
    with open(conllu_path, encoding="utf-8") as f:
        return conllu_lib.parse(f.read())

def run_inference(parser, conllu_path, pred_conllu_path, force_rerun=False):
    """
    Write predicted CoNLL-U using parser.predict() — guaranteed correct output.
    Uses cached file if force_rerun=False and file already exists.
    Returns parsed predicted sentences.
    """
    if (not force_rerun
            and os.path.exists(pred_conllu_path)
            and os.path.getsize(pred_conllu_path) > 0):
        print(f"    Cached: {os.path.basename(pred_conllu_path)}")
        with open(pred_conllu_path, encoding="utf-8") as f:
            return conllu_lib.parse(f.read())

    parser.predict(conllu_path, pred=pred_conllu_path, verbose=False)

    with open(pred_conllu_path, encoding="utf-8") as f:
        pred_sents = conllu_lib.parse(f.read())

    print(f"    Saved: {os.path.basename(pred_conllu_path)} ({len(pred_sents)} sentences)")
    return pred_sents

def score_predictions(gold_sents, pred_sents):
    """
    Token-level UAS/LAS + sentence-level boolean arrays for McNemar's tests.
    gold_sents and pred_sents must both be freshly parsed from their respective
    files — never reuse objects that were modified in place.
    """
    total = correct_arc = correct_both = 0
    sent_uas, sent_las = [], []
    for gold, pred in zip(gold_sents, pred_sents):
        gt = [t for t in gold if isinstance(t["id"], int)]
        pt = [t for t in pred if isinstance(t["id"], int)]
        n = len(gt)
        if n == 0:
            continue
        arc_ok  = sum(1 for g, p in zip(gt, pt) if g["head"] == p["head"])
        both_ok = sum(1 for g, p in zip(gt, pt)
                      if g["head"] == p["head"] and g["deprel"] == p["deprel"])
        total        += n
        correct_arc  += arc_ok
        correct_both += both_ok
        sent_uas.append(arc_ok  == n)
        sent_las.append(both_ok == n)
    uas = round(100 * correct_arc  / total, 2) if total else 0.0
    las = round(100 * correct_both / total, 2) if total else 0.0
    return uas, las, np.array(sent_uas, dtype=bool), np.array(sent_las, dtype=bool)

def per_relation_las(gold_sents, pred_sents):
    counts = defaultdict(lambda: [0, 0])
    for gold, pred in zip(gold_sents, pred_sents):
        gt = [t for t in gold if isinstance(t["id"], int)]
        pt = [t for t in pred if isinstance(t["id"], int)]
        for g, p in zip(gt, pt):
            rel = g["deprel"]
            counts[rel][1] += 1
            if g["head"] == p["head"] and g["deprel"] == p["deprel"]:
                counts[rel][0] += 1
    return {
        rel: {"correct": c, "total": t,
              "LAS": round(100 * c / t, 2) if t else 0.0}
        for rel, (c, t) in sorted(counts.items(), key=lambda x: -x[1][1])
    }

def evaluate_and_save(parser, system_name, conllu_path, test_name, gold_sents,
                      force_rerun=False):
    """
    Run inference, score with parser.evaluate() for UAS/LAS accuracy,
    save CoNLL-U predictions and per-sentence npy for McNemar's tests.

    UAS/LAS come from parser.evaluate() (not our manual loop) because
    parser.predict() / parser.evaluate() are guaranteed internally consistent.
    score_predictions() is used only to produce the sentence-level boolean
    arrays needed for McNemar's significance tests.
    """
    safe        = system_name.replace(" ", "_")
    pred_conllu = os.path.join(PRED_DIR, f"{safe}_{test_name}.conllu")
    pred_npy    = os.path.join(PRED_DIR, f"{safe}_{test_name}_sent_las.npy")

    print(f"  [{system_name}] → {test_name}  (force_rerun={force_rerun})")
    pred_sents = run_inference(parser, conllu_path, pred_conllu, force_rerun=force_rerun)

    # Authoritative UAS/LAS from SuPar's own evaluator
    _, metric = parser.evaluate(conllu_path, verbose=False)
    uas = round(metric.uas * 100, 2)
    las = round(metric.las * 100, 2)

    # Sentence-level arrays for McNemar's — load gold fresh to avoid mutation
    gold_fresh = load_gold(conllu_path)
    _, _, s_uas, s_las = score_predictions(gold_fresh, pred_sents)
    np.save(pred_npy, s_las)

    print(f"    UAS {uas:.2f}  LAS {las:.2f}  npy saved: {os.path.basename(pred_npy)}")
    return {
        "system"      : system_name,
        "test"        : test_name,
        "UAS"         : uas,
        "LAS"         : las,
        "pred_conllu" : os.path.basename(pred_conllu),
        "pred_npy"    : os.path.basename(pred_npy),
        "n_sentences" : len(pred_sents),
    }

print("Helpers defined.")


Helpers defined.


---
## 4 — Train Helper

In [5]:
from supar import CRF2oDependencyParser
import supar
print(f"SuPar: {supar.__version__}")

def train_parser(tb_name):
    """
    Train CRF2oDependencyParser on the given treebank.
    Skips training if checkpoint already exists on Drive.

    Per-treebank LR:
      BOUN: 1e-3  IMST: 2e-3  (empirically determined)
    """
    splits     = BOUN if tb_name == "BOUN" else IMST
    model_path = MODEL_PATHS[tb_name]
    lr         = LR_PER_TB[tb_name]

    print(f"\n{'='*60}")
    print(f" {tb_name} | LR={lr}  BERT_LR={BERT_LR}  EPOCHS={MAX_EPOCHS}  PATIENCE={PATIENCE}")
    print(f"{'='*60}")

    if os.path.exists(model_path):
        print("Checkpoint found — skipping training.")
        print(f"Delete {model_path} from Drive to retrain from scratch.")
        parser = CRF2oDependencyParser.load(model_path)
        parser.args.update(SUPAR_EXTRA)
        return parser

    print("No checkpoint — training from scratch.")
    parser = CRF2oDependencyParser.build(
        path    = model_path,
        build   = True,
        encoder = "bert",
        bert    = BERTURK_NAME,
        train   = splits["train"],
    )
    parser.train(
        train        = splits["train"],
        dev          = splits["dev"],
        test         = splits["test"],
        batch_size   = BATCH_SIZE,
        epochs       = MAX_EPOCHS,
        patience     = PATIENCE,
        lr           = lr,
        lr_rate      = BERT_LR,
        warmup       = WARMUP,
        weight_decay = WEIGHT_DECAY,
        eps          = EPS,
        verbose      = True,
        checkpoint   = False,
        seed         = SEED,
        device       = "0",
    )
    parser.args.update(SUPAR_EXTRA)
    return parser

print("train_parser() ready.")


SuPar: 1.1.4
train_parser() ready.


---
## 5 — Train BOUN Baseline

LR=1e-3. Expected runtime: ~25 min on A100.

In [6]:
parser_boun = train_parser("BOUN")


 BOUN | LR=0.001  BERT_LR=2e-05  EPOCHS=30  PATIENCE=10
Checkpoint found — skipping training.
Delete /content/drive/MyDrive/58T - Application Project/models/baseline_boun.pt from Drive to retrain from scratch.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/445M [00:00<?, ?B/s]

---
## 6 — Train IMST Baseline

LR=2e-3. Expected runtime: ~5 min on A100.

In [8]:
parser_imst = train_parser("IMST")


 IMST | LR=0.002  BERT_LR=2e-05  EPOCHS=30  PATIENCE=10
Checkpoint found — skipping training.
Delete /content/drive/MyDrive/58T - Application Project/models/baseline_imst.pt from Drive to retrain from scratch.


EOFError: Ran out of input

---
## 7 — Evaluate

`force_rerun=True` ensures stale prediction files from earlier runs are overwritten.

In [ ]:
import json

gold = {
    "BOUN": load_gold(BOUN["test"]),
    "IMST": load_gold(IMST["test"]),
    "PUD" : load_gold(PUD["test"]),
}

results = []

# ── BOUN parser ───────────────────────────────────────────────────────────────
if not os.path.exists(MODEL_PATHS["BOUN"]):
    raise FileNotFoundError("baseline_boun.pt not found. Run section 5 first.")

print("Loading baseline_boun.pt...")
parser_boun = CRF2oDependencyParser.load(MODEL_PATHS["BOUN"])
parser_boun.args.update(SUPAR_EXTRA)

for test_name, test_path in [("BOUN", BOUN["test"]), ("PUD", PUD["test"])]:
    r = evaluate_and_save(parser_boun, "baseline_boun", test_path, test_name,
                          gold[test_name], force_rerun=True)
    results.append(r)

del parser_boun
torch.cuda.empty_cache()

# ── IMST parser ───────────────────────────────────────────────────────────────
if not os.path.exists(MODEL_PATHS["IMST"]):
    raise FileNotFoundError("baseline_imst.pt not found. Run section 6 first.")

print("\nLoading baseline_imst.pt...")
parser_imst = CRF2oDependencyParser.load(MODEL_PATHS["IMST"])
parser_imst.args.update(SUPAR_EXTRA)

for test_name, test_path in [("IMST", IMST["test"]), ("PUD", PUD["test"])]:
    r = evaluate_and_save(parser_imst, "baseline_imst", test_path, test_name,
                          gold[test_name], force_rerun=True)
    results.append(r)

del parser_imst
torch.cuda.empty_cache()

results_path = os.path.join(RESULTS_DIR, "baseline_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved: {results_path}")


Loading baseline_boun.pt...
  [baseline_boun] → BOUN  (force_rerun=True)


/usr/local/lib/python3.12/dist-packages/supar/utils/fn.py:155: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  out_tensor[i][[slice(-i, None) if padding_side == 'left' else slice(0, i) for i in tensor.size()]] = tensor
/usr/local/lib/python3.12/dist-packages/supar/utils/fn.py:155: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:34

    Saved: baseline_boun_BOUN.conllu (979 sentences)
    UAS 77.26  LAS 61.96  npy saved: baseline_boun_BOUN_sent_las.npy
  [baseline_boun] → PUD  (force_rerun=True)
    Saved: baseline_boun_PUD.conllu (1000 sentences)
    UAS 76.26  LAS 60.41  npy saved: baseline_boun_PUD_sent_las.npy

Loading baseline_imst.pt...


EOFError: Ran out of input

---
## 8 — Results Table

In [ ]:
print(f"{'System':<20} {'Test':<6} {'UAS':>7} {'LAS':>7}  Predictions")
print("-" * 75)
for r in results:
    print(f"  {r['system']:<18} {r['test']:<6} {r['UAS']:>7.2f} {r['LAS']:>7.2f}  {r['pred_conllu']}")


---
## 9 — Per-Relation LAS Breakdown

In [ ]:
import json

pred_boun_path = os.path.join(PRED_DIR, "baseline_boun_BOUN.conllu")

if not os.path.exists(pred_boun_path):
    print("Prediction file not found — run section 7 (Evaluate) first.")
else:
    gold_boun = load_gold(BOUN["test"])
    with open(pred_boun_path, encoding="utf-8") as f:
        pred_boun_sents = conllu_lib.parse(f.read())

    breakdown = per_relation_las(gold_boun, pred_boun_sents)

    br_path = os.path.join(RESULTS_DIR, "baseline_boun_BOUN_per_rel.json")
    with open(br_path, "w") as f:
        json.dump(breakdown, f, indent=2)
    print(f"Per-relation breakdown saved: {br_path}")
    print()
    print(f"{'Relation':<15} {'Count':>7} {'LAS':>7}")
    print("-" * 32)
    for rel, vals in list(breakdown.items())[:20]:
        print(f"  {rel:<13} {vals['total']:>7} {vals['LAS']:>7.2f}")


---
## 10 — Verify Saved Files

In [ ]:
import os

expected_files = [
    (MODEL_DIR,   "baseline_boun.pt"),
    (MODEL_DIR,   "baseline_imst.pt"),
    (PRED_DIR,    "baseline_boun_BOUN.conllu"),
    (PRED_DIR,    "baseline_boun_BOUN_sent_las.npy"),
    (PRED_DIR,    "baseline_boun_PUD.conllu"),
    (PRED_DIR,    "baseline_boun_PUD_sent_las.npy"),
    (PRED_DIR,    "baseline_imst_IMST.conllu"),
    (PRED_DIR,    "baseline_imst_IMST_sent_las.npy"),
    (PRED_DIR,    "baseline_imst_PUD.conllu"),
    (PRED_DIR,    "baseline_imst_PUD_sent_las.npy"),
    (RESULTS_DIR, "baseline_results.json"),
    (RESULTS_DIR, "baseline_boun_BOUN_per_rel.json"),
]

print(f"{'Status':<8} {'Size':>12}  {'Dir':<14} File")
print("-" * 85)
all_ok = True
for directory, filename in expected_files:
    path   = os.path.join(directory, filename)
    exists = os.path.exists(path)
    size   = f"{os.path.getsize(path):,} B" if exists else "-"
    status = "OK" if exists else "MISSING"
    folder = os.path.basename(directory)
    if not exists:
        all_ok = False
    print(f"  {status:<6} {size:>12}  {folder:<14} {filename}")

print()
if all_ok:
    print("All files present. 02_baseline complete.")
else:
    print("Some files missing — rerun the relevant sections above.")
